[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C62_Coding_Interview_Course/02_hash_sort_search/02_hash_sort_search.ipynb)

# 02 · 哈希、排序与二分（三套二分模板 / 答案空间二分 / Top-K 三解法 / 排序确定性）

目标：把这三样东西从「大概会」变成「能在压力下一次写对，并说清为什么对」。

本 notebook 你会亲手实现：
1. **玩具哈希表**，把「最坏 O(n)」真的跑出来；再验证 Python dict 的三个坑（`hash(1)==hash(1.0)==hash(True)`、浮点 key、可哈希性）
2. **扩容的均摊分析**：单次 O(n)，均摊 O(1) —— 并看清它对延迟分布的影响
3. **排序稳定性**：分次排序实现多关键字；**同一批框、同样的分数，tie-break 不同 → NMS 输出不同框**
4. **三套二分模板** + 循环不变量断言 + 与 `bisect` 的 2000 组随机对拍 + 换算表
5. **把死循环跑出来**：区间长度停止下降的完整轨迹；以及求右边界为什么必须用上取整 mid
6. **在答案上二分**：LC410 与 O(n²m) DP 对拍；再用同一套模板解「给定召回下限求最高 score 阈值」
7. **Top-K 三解法**（全排序 / 堆 / quickselect）的对拍与计时，以及元素访问次数的线性性验证
8. **10 道经典题的暴力解 vs 最优解随机对拍**，每题标注频率与层级

> 心智模型：**二分的前提不是「数组有序」，是「存在一个可切分的单调谓词」；
> 而贪心算法的输出永远依赖排序顺序，所以排序键必须是全序。**

## 1 · 哈希表：均摊 O(1) 的边界在哪

先用一个链地址法的玩具哈希表，把「哈希函数退化 ⇒ 查找变 O(n)」跑成数字。
用**探测次数**（确定性）而不是墙钟时间（有噪声）做断言依据。

In [ ]:
import sys, time, random, bisect, heapq
from collections import Counter, defaultdict
import numpy as np

print('Python', sys.version.split()[0], '| numpy', np.__version__)


class ToyHash:
    """链地址法哈希表（教学模型；CPython 实际用开放寻址）。probes 累计探测次数。"""
    def __init__(self, cap=64, hashfn=hash):
        self.cap, self.hashfn = cap, hashfn
        self.buckets = [[] for _ in range(cap)]
        self.probes = 0

    def _idx(self, k):
        return self.hashfn(k) % self.cap

    def put(self, k, v):
        b = self.buckets[self._idx(k)]
        for i, (kk, _) in enumerate(b):
            self.probes += 1
            if kk == k:
                b[i] = (k, v); return
        b.append((k, v))

    def get(self, k):
        for kk, vv in self.buckets[self._idx(k)]:
            self.probes += 1
            if kk == k:
                return vv
        raise KeyError(k)


N = 64
good = ToyHash(cap=64)                          # 好哈希：key 均匀散到 64 个桶
bad  = ToyHash(cap=64, hashfn=lambda k: 0)      # 坏哈希：所有 key 撞进同一个桶
for i in range(N):
    good.put(i, i); bad.put(i, i)
good.probes = bad.probes = 0                    # 清零，只统计查找
for i in range(N):
    good.get(i); bad.get(i)

print(f'好哈希：{N} 次查找共探测 {good.probes:>5} 次  → 平均 {good.probes / N:6.2f}')
print(f'坏哈希：{N} 次查找共探测 {bad.probes:>5} 次  → 平均 {bad.probes / N:6.2f}')
assert good.probes == N                         # 每桶恰好 1 个元素
assert bad.probes == N * (N + 1) // 2           # 第 i 次查找扫 i+1 个 → 等差和
print('\n✅ 「最坏 O(n)」不是理论玩具：哈希函数一退化，平均探测次数从 1 变成 (n+1)/2 =',
      (N + 1) / 2)

In [ ]:
# ── Python dict 的三个坑（面试里被追问「dict 有什么坑」时的标准答案）──

# 坑 1：hash(1) == hash(1.0) == hash(True)
assert hash(1) == hash(1.0) == hash(True)
d = {1: 'x', 1.0: 'y', True: 'z'}
print("{1:'x', 1.0:'y', True:'z'} =", d, '  ← 三个 key 合并成了一个')
assert len(d) == 1 and d[1] == 'z' and d[True] == 'z'

# 坑 2：浮点数做 key
d2 = {0.3: 'ok'}
print('0.1 + 0.2 =', repr(0.1 + 0.2), ' → 用它查 d2 会 KeyError')
assert (0.1 + 0.2) not in d2

nan = float('nan')
d3 = {nan: 'reachable-by-identity'}
assert d3[nan] == 'reachable-by-identity'   # 同一个对象：dict 先比 is，短路成功
assert float('nan') not in d3               # 另一个 nan 对象：is 不成立、== 也不成立
print('nan 作 key：同一个对象查得到，另一个 nan 查不到 —— 因为 nan != nan')

# 坑 3：可哈希性是递归计算的
assert isinstance(hash((1, 2)), int)
for bad_key in ([1, 2], (1, [2]), {1: 2}):
    try:
        hash(bad_key)
        raise AssertionError('应该抛 TypeError：' + repr(bad_key))
    except TypeError:
        pass
print('✅ list / 含 list 的 tuple / dict 都不可哈希（哈希递归到每个元素）')

# 附：分桶评测绝不能用 round 后的浮点当 key —— 舍入方向随数值不规则翻转
print('\nround(0.035, 2) =', round(0.035, 2), '  ← 向上')
print('round(0.045, 2) =', round(0.045, 2), '  ← 向下（两个相邻的中点值被舍进了同一个桶）')
print('round(2.675, 2) =', round(2.675, 2), '  ← 著名的「应该是 2.68」')
assert round(0.035, 2) == 0.04 and round(0.045, 2) == 0.04 and round(2.675, 2) == 2.67
ups   = [x / 200 for x in range(1, 40, 2) if abs(round(x / 200, 2) - (x / 200 + 0.005)) < 1e-12]
downs = [x / 200 for x in range(1, 40, 2) if abs(round(x / 200, 2) - (x / 200 - 0.005)) < 1e-12]
print('向上舍入的中点值：', ups[:5])
print('向下舍入的中点值：', downs[:5])
assert ups and downs, '两个方向都存在 ⇒ 舍入方向不可预测'
print('   ↑ 因为这些十进制中点值在二进制里根本不是中点，方向由二进制表示决定，不规则。')

In [ ]:
# ── 均摊分析：翻倍扩容的总重建代价是等比级数 ──
def rebuild_cost(n, load=2/3, start=8):
    """模拟 CPython 的 dict 扩容：装载因子超过 load 就翻倍并重建全部条目。"""
    cap, size, total, events = start, 0, 0, []
    for _ in range(n):
        if size + 1 > load * cap:
            cap *= 2
            total += size                 # 一次重建 = 搬运当前全部元素
            events.append((size, cap))
        size += 1
    return total, total / n, events

for n in (10**3, 10**4, 10**5, 10**6):
    tot, amo, ev = rebuild_cost(n)
    print(f'n={n:>8}  扩容 {len(ev):>2} 次  总搬运 {tot:>9}  均摊每次插入 {amo:5.3f}')
    assert amo < 2.0, '均摊代价必须是 O(1) 的小常数'

_, _, ev = rebuild_cost(10**4)
print('\n第 10^4 次插入前的扩容时刻（size → 新容量）：', ev[-4:])
print('✅ 单次扩容 O(n)，均摊 O(1)。')
print('   但注意：**均摊 O(1) 意味着某一次插入会突然很慢** ——')
print('   车端后处理这类每帧都要跑的关键路径上，要预分配容量，别让它在帧内扩容。')

## 2 · 排序：稳定性的两个用途

第一个是语法层的（多关键字），第二个是系统层的（**输出确定性**）——后者才是面试的重点。

In [ ]:
# ── 用途一：分次排序实现「主关键字降序 + 次关键字升序」──
recs = [('bravo', 2), ('alpha', 1), ('carol', 2), ('alpha', 3), ('bravo', 1)]

two_pass = sorted(recs, key=lambda r: r[1])       # ② 先排次关键字
two_pass = sorted(two_pass, key=lambda r: r[0])   # ① 再排主关键字（稳定 ⇒ 次序被保留）
one_pass = sorted(recs, key=lambda r: (r[0], r[1]))
print('分次排序 =', two_pass)
print('元组 key =', one_pass)
assert two_pass == one_pass
print('✅ 「先排次关键字，再排主关键字」只有在排序稳定时才成立')
print('   （主关键字要降序、次关键字是字符串没法取负时，这是唯一的写法）')

# Timsort 的自适应性：几乎有序的输入远快于随机输入
n = 200_000
near_sorted = list(range(n)); random.Random(0).shuffle(near_sorted[:200])
shuffled = list(range(n)); random.Random(0).shuffle(shuffled)
t0 = time.perf_counter(); sorted(near_sorted); t_near = time.perf_counter() - t0
t0 = time.perf_counter(); sorted(shuffled);    t_rand = time.perf_counter() - t0
print(f'\n几乎有序 {t_near*1000:7.2f} ms   完全乱序 {t_rand*1000:7.2f} ms   '
      f'比值 {t_rand/max(t_near,1e-9):.1f}x')
print('   ↑ Timsort 检测已有的升序 run 并直接归并 → 近乎 O(n)')

In [ ]:
# ── 用途二（重点）：NMS 的输出依赖排序的 tie-break ──
def iou_1_to_n(box, boxes):
    x1 = np.maximum(box[0], boxes[:, 0]); y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2]); y2 = np.minimum(box[3], boxes[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    a0 = (box[2] - box[0]) * (box[3] - box[1])
    a1 = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return inter / np.maximum(a0 + a1 - inter, 1e-9)

def nms(boxes, order, iou_thr=0.5):
    """贪心 NMS。order 是已排好序的下标序列 —— tie-break 由调用者决定。"""
    order, keep = list(order), []
    while order:
        i = order.pop(0); keep.append(i)
        if not order:
            break
        ious = iou_1_to_n(boxes[i], boxes[order])
        order = [j for j, u in zip(order, ious) if u <= iou_thr]
    return keep

# A、B 高度重叠且分数完全相同；C 在远处
BX = np.array([[100., 200., 140., 240.],      # A
               [103., 204., 143., 244.],      # B
               [600., 300., 660., 360.]])     # C
SC = np.array([0.90, 0.90, 0.75])
u_ab = float(iou_1_to_n(BX[0], BX[1:2])[0])
print(f'IoU(A,B) = 1332/1868 = {u_ab:.4f}  > 0.5 → A、B 必有一个被抑制')
assert abs(u_ab - 1332 / 1868) < 1e-12

keep_ab = nms(BX, [0, 1, 2])      # A 排在 B 前
keep_ba = nms(BX, [1, 0, 2])      # B 排在 A 前
print('排序 [A,B,C] → 保留下标', keep_ab, ' 框中心 x =', BX[keep_ab[0]][0])
print('排序 [B,A,C] → 保留下标', keep_ba, ' 框中心 x =', BX[keep_ba[0]][0])
assert keep_ab == [0, 2] and keep_ba == [1, 2] and keep_ab != keep_ba
print('\n❗ 同一批框、同样的分数，只因 tie-break 不同，输出的框就差了 3 个像素。')
print('   分数并列一点都不罕见：INT8 量化后分类头只有 256 个可能取值（见 C60）。')

In [ ]:
# ── 修法：把排序键做成全序（total order），并列就不存在了 ──
def order_by(scores, boxes, tiebreak=True):
    n = len(scores)
    if tiebreak:   # (-分数, x1, y1, 原下标) —— 任意两个不同元素的键都不相等
        key = lambda i: (-float(scores[i]), float(boxes[i, 0]), float(boxes[i, 1]), i)
    else:          # 只有分数 → 存在并列 → 结果依赖实现
        key = lambda i: -float(scores[i])
    return sorted(range(n), key=key)

perm = [2, 1, 0]                                   # 同一批框，换个输入顺序
BXp, SCp = BX[perm], SC[perm]

for tb in (True, False):
    k1 = sorted(nms(BX, order_by(SC, BX, tb)))
    k2 = sorted(perm[j] for j in nms(BXp, order_by(SCp, BXp, tb)))
    tag = '全序键 ' if tb else '仅分数键'
    print(f'{tag}: 原顺序输入 → {k1} ；打乱输入 → {k2} ；一致？{k1 == k2}')

k1 = sorted(nms(BX, order_by(SC, BX, True)))
k2 = sorted(perm[j] for j in nms(BXp, order_by(SCp, BXp, True)))
assert k1 == k2 == [0, 2], (k1, k2)
print('\n✅ 全序排序键 ⇒ 输出与输入顺序无关；此时「排序是否稳定」不再重要，因为不会出现并列。')

## 3 · 三套二分模板与循环不变量

每个模板里都写了 `assert` 形式的**循环不变量**（教学用；生产代码里当然要关掉）。
不变量成立 + 区间长度严格递减 ⇒ 正确性与终止性都是被**证明**的，不是被测试的。

In [ ]:
def bs_exact(a, t):
    """模板 A · 闭区间 [lo, hi]。不变量 I_A：若 t 存在于 a 中，其下标 ∈ [lo, hi]。"""
    lo, hi = 0, len(a) - 1
    while lo <= hi:                       # 闭区间非空
        mid = lo + (hi - lo) // 2         # 这样写而不是 (lo+hi)//2：C++/Java 里防溢出
        if a[mid] == t:
            return mid
        if a[mid] < t:
            lo = mid + 1                  # a[:mid+1] 全 < t → I_A 保持
        else:
            hi = mid - 1                  # a[mid:] 全 > t → I_A 保持
    return -1                             # 区间空 ⇒ 由 I_A，t 不存在


def bs_left(a, t, check_inv=True):
    """模板 B · 半开区间 [lo, hi) = bisect_left。
       不变量 I_B：a[:lo] 全 < t  且  a[hi:] 全 >= t。"""
    lo, hi = 0, len(a)
    while lo < hi:
        if check_inv:
            assert all(x < t for x in a[:lo]) and all(x >= t for x in a[hi:]), 'I_B 破了'
        mid = lo + (hi - lo) // 2         # 下取整 ⇒ lo <= mid < hi
        if a[mid] < t:
            lo = mid + 1
        else:
            hi = mid                      # 注意是 mid，不是 mid-1
    return lo


def bs_right(a, t, check_inv=True):
    """模板 C · 半开区间 [lo, hi) = bisect_right。
       不变量 I_C：a[:lo] 全 <= t  且  a[hi:] 全 > t。与模板 B 只差一个 `=`。"""
    lo, hi = 0, len(a)
    while lo < hi:
        if check_inv:
            assert all(x <= t for x in a[:lo]) and all(x > t for x in a[hi:]), 'I_C 破了'
        mid = lo + (hi - lo) // 2
        if a[mid] <= t:
            lo = mid + 1
        else:
            hi = mid
    return lo


demo = [1, 3, 3, 3, 7]
print('a =', demo)
print('bs_left (a,3) =', bs_left(demo, 3), '  （第一个 >= 3）')
print('bs_right(a,3) =', bs_right(demo, 3), '  （第一个 > 3）')
print('出现次数      =', bs_right(demo, 3) - bs_left(demo, 3))
assert (bs_left(demo, 3), bs_right(demo, 3)) == (1, 4)

In [ ]:
# ── 与 bisect 的 2000 组随机对拍，同时验证「换算表」──
rng_bs = random.Random(0)
for _ in range(2000):
    n = rng_bs.randint(0, 12)
    a = sorted(rng_bs.randint(0, 6) for _ in range(n))
    t = rng_bs.randint(-1, 7)

    assert bs_left(a, t)  == bisect.bisect_left(a, t),  (a, t)
    assert bs_right(a, t) == bisect.bisect_right(a, t), (a, t)

    i = bs_exact(a, t)
    assert (i == -1) == (t not in a), (a, t, i)
    if i != -1:
        assert a[i] == t

    # 换算表：只背 bs_left / bs_right，其余全部现推
    assert bs_right(a, t) - bs_left(a, t) == a.count(t)
    assert bs_right(a, t) - 1 == max([j for j, x in enumerate(a) if x <= t], default=-1)
    assert bs_left(a, t) - 1  == max([j for j, x in enumerate(a) if x < t],  default=-1)
    assert bs_right(a, t) == len(a) - len([x for x in a if x > t])

print('✅ 2000 组随机数组（含空数组、大量重复、目标不存在）：')
print('   三套模板与 bisect 完全一致，四条换算关系全部成立。')
print('   ⇒ 面试里只背 bs_left 一套，其余现推 —— 少一套模板就少一类 bug。')

## 4 · 把死循环跑出来

区间长度是二分的**度量函数（variant）**。死循环的定义就是它不再严格递减。

In [ ]:
def bs_left_buggy(a, t, max_steps=64):
    """错误示范：半开区间里把 lo = mid + 1 写成了 lo = mid。
       返回 (结果, 步数, 是否撞上步数上界)。"""
    lo, hi = 0, len(a)
    for step in range(1, max_steps + 1):
        if lo >= hi:
            return lo, step - 1, False
        mid = lo + (hi - lo) // 2
        if a[mid] < t:
            lo = mid                      # ← 少了 +1：更新没有「跨过 mid」
        else:
            hi = mid
    return None, max_steps, True          # 撞上界 = 真实场景下的死循环

arr = [1, 3, 5]
res, steps, looped = bs_left_buggy(arr, 5)
print(f'bs_left_buggy([1,3,5], 5) → 结果 {res}，步数 {steps}，卡死 {looped}')
assert looped is True and res is None
assert bs_left(arr, 5) == 2               # 正确实现 2 步就出来了

# 逐轮打印度量函数
print('\n轮次   lo  hi  mid  a[mid]   更新         区间长度')
lo, hi = 0, len(arr)
for step in range(1, 7):
    if lo >= hi:
        break
    mid = lo + (hi - lo) // 2
    old_lo, old_hi = lo, hi
    if arr[mid] < 5:
        lo, act = mid, 'lo = mid  (错)'
    else:
        hi, act = mid, 'hi = mid  (对)'
    old = old_hi - old_lo
    flag = '' if hi - lo < old else '   ← 没有下降！'
    print(f'{step:^5}{old_lo:>4}{old_hi:>4}{mid:>5}{arr[mid]:>8}   {act:<14}{old} → {hi-lo}{flag}')
print('\n✅ 根因：区间长度为 1 时下取整让 mid == lo，而 `lo = mid` 没有跨过 mid。')
print('   通用判据：**mid 落在哪一半，那一半的更新就必须跨过 mid**。')

In [ ]:
# ── 求右边界的唯一安全组合：上取整 mid + lo = mid ──
def bs_last_le(a, t):
    """最后一个 <= t 的下标（不存在返回 -1）。闭区间 [lo, hi]，答案最终落在 lo。
       不变量：a[lo] <= t（lo = -1 是虚拟哨兵，视为恒成立）且 a[hi+1:] 全 > t。"""
    lo, hi = -1, len(a) - 1
    while lo < hi:
        mid = lo + (hi - lo + 1) // 2     # **上取整**：把 mid 顶到右半边
        if a[mid] <= t:
            lo = mid                      # 保留 mid；因为 mid > lo，区间仍然缩小
        else:
            hi = mid - 1
    return lo

a2 = [1, 3, 3, 3, 7]
for t in range(-1, 9):
    assert bs_last_le(a2, t) == bisect.bisect_right(a2, t) - 1, t
print('a2 =', a2)
print('t        :', list(range(-1, 9)))
print('last_le  :', [bs_last_le(a2, t) for t in range(-1, 9)])
print('✅ 上取整 mid 配 `lo = mid` 才不会死循环；下取整配 `lo = mid` 必死。')
print('   —— 这就是为什么本课建议：只用半开区间 + bs_left，右边界一律用换算表推。')

## 5 · 在答案上二分：最小化最大值

三个识别信号：① 题面是「最小化最大值 / 至少需要多少」；② `check(x)` 好写（O(n) 贪心）；
③ **`check` 单调**。第三条必须先说出来，否则就是背题。

In [ ]:
def split_array_bs(a, m):
    """LC410：把 a 分成 m 个连续子数组，最小化「各段和的最大值」。O(n log(sum a))。"""
    def parts(cap):
        # check：每段和不超过 cap 时，贪心地尽量多塞，最少需要几段
        cnt, cur = 1, 0
        for x in a:
            if cur + x <= cap:
                cur += x
            else:
                cnt += 1; cur = x
        return cnt

    lo, hi = max(a), sum(a)     # lo：任何一段至少要装得下最大元素；hi：全放一段一定可行
    while lo < hi:              # 不变量：答案 ∈ [lo, hi]
        mid = lo + (hi - lo) // 2
        if parts(mid) <= m:
            hi = mid            # mid 可行 ⇒ 答案 <= mid（保留 mid）
        else:
            lo = mid + 1        # mid 不可行 ⇒ 答案 > mid
    return lo


def split_array_dp(a, m):
    """暴力对照：区间 DP，O(n²m)。dp[k][i] = 前 i 个元素分 k 段时，最大段和的最小值。"""
    n = len(a); INF = float('inf')
    pre = [0] * (n + 1)
    for i, x in enumerate(a):
        pre[i + 1] = pre[i] + x
    dp = [[INF] * (n + 1) for _ in range(m + 1)]
    dp[0][0] = 0
    for k in range(1, m + 1):
        for i in range(k, n + 1):
            for j in range(k - 1, i):
                if dp[k - 1][j] < INF:
                    dp[k][i] = min(dp[k][i], max(dp[k - 1][j], pre[i] - pre[j]))
    return dp[m][n]


print('split_array([7,2,5,10,8], 2) =', split_array_bs([7, 2, 5, 10, 8], 2), '（期望 18）')
assert split_array_bs([7, 2, 5, 10, 8], 2) == 18

rng_sa = random.Random(7)
for _ in range(200):
    n = rng_sa.randint(1, 7)
    a = [rng_sa.randint(0, 12) for _ in range(n)]
    m = rng_sa.randint(1, n)
    assert split_array_bs(a, m) == split_array_dp(a, m), (a, m)
print('✅ 200 组随机数据：答案空间二分与 O(n²m) DP 完全一致')
print('   单调性论证：cap 越大 → 需要的段数单调不增 → parts(cap) <= m 是单调谓词。')

In [ ]:
# ── 同一套模板的工程同构：给定召回下限，求最高可用的 score 阈值 ──
# （这正是 C55-05 的「工作点选择」；在代码上它就是一次答案空间二分）
rng_np = np.random.default_rng(0)
scores_tp = rng_np.beta(5, 2, size=400)          # 400 个真实目标各自的检出分数

def recall_at(thr):
    return float((scores_tp >= thr).mean())      # thr 越大 → 召回越低（单调不增）

TARGET = 0.95
lo_f, hi_f = 0.0, 1.0                            # 不变量：lo 可行、hi 不可行
for _ in range(100):                             # 浮点二分：**固定迭代次数**，绝不 while
    mid = (lo_f + hi_f) / 2
    if recall_at(mid) >= TARGET:
        lo_f = mid
    else:
        hi_f = mid

print(f'满足 recall >= {TARGET} 的最高阈值 ≈ {lo_f:.6f}')
print(f'  该阈值处 recall = {recall_at(lo_f):.4f}；再抬一点点 recall = {recall_at(hi_f):.4f}')
assert recall_at(lo_f) >= TARGET and recall_at(hi_f) < TARGET
print(f'区间已缩到 {hi_f - lo_f:.3e}（100 次迭代 ≈ 2^-100），远超 double 精度')
print('\n✅ 浮点答案空间不要用 `while lo < hi`（浮点相等几乎永不成立 → 死循环），')
print('   用固定 100 次迭代：既够精确，又绝对不会卡住。')

## 6 · Top-K 的三种解法：对拍 + 计时

重点不是「哪个快」，而是**渐进复杂度 ≠ 实际耗时**，以及三种解法各自买到了什么。

In [ ]:
def topk_sort(a, k):
    """① 全排序 O(n log n)，空间 O(n)，输出天然有序。"""
    return sorted(a, reverse=True)[:k]


def topk_heap(a, k):
    """② 大小为 k 的最小堆：堆顶就是「当前第 k 大」，比它小的直接丢。
       O(n log k)，空间 O(k)，**只遍历一次 ⇒ 流式可用**。"""
    h = []
    for x in a:
        if len(h) < k:
            heapq.heappush(h, x)
        elif x > h[0]:
            heapq.heapreplace(h, x)      # 弹出最小、压入 x，一次堆调整
    return sorted(h, reverse=True)


_QS_VISITS = 0

def quickselect_kth(a, k, rnd):
    """③ 返回第 k 小（k 从 0 开始）的值。随机主元 + 三路划分，期望 O(n)。
       划分不变量：[lo,i) < p，[i,t) == p，(j,hi] > p；答案下标 k 始终落在 [lo,hi]。"""
    global _QS_VISITS
    a = list(a)                          # 不修改调用方的数据
    lo, hi = 0, len(a) - 1
    while True:
        p = a[rnd.randint(lo, hi)]       # 随机主元：否则有序输入会退化成 O(n²)
        i, j, t = lo, hi, lo
        while t <= j:                    # 荷兰国旗三路划分
            _QS_VISITS += 1
            if a[t] < p:
                a[i], a[t] = a[t], a[i]; i += 1; t += 1
            elif a[t] > p:
                a[t], a[j] = a[j], a[t]; j -= 1
            else:
                t += 1
        if k < i:
            hi = i - 1                   # 答案在左段
        elif k > j:
            lo = j + 1                   # 答案在右段
        else:
            return p                     # 落在「等于主元」的那段


def topk_quickselect(a, k, rnd):
    if k <= 0:
        return []
    v = quickselect_kth(a, len(a) - k, rnd)      # 第 k 大 = 第 (n-k) 小
    out = [x for x in a if x > v]
    out += [v] * (k - len(out))                  # 用主元值补齐（处理重复元素）
    return sorted(out, reverse=True)


rnd_tk = random.Random(1)
for _ in range(300):
    n = rnd_tk.randint(1, 40)
    a = [rnd_tk.randint(0, 15) for _ in range(n)]     # 大量重复，专门考验三路划分
    k = rnd_tk.randint(1, n)
    r1, r2, r3 = topk_sort(a, k), topk_heap(a, k), topk_quickselect(a, k, rnd_tk)
    assert r1 == r2 == r3, (a, k, r1, r2, r3)
print('✅ 300 组随机对拍（含大量重复元素）：三种解法输出完全一致')

In [ ]:
# ── 计时与「元素访问次数」──
N_BIG, K = 100_000, 100
big = [rnd_tk.randrange(10**9) for _ in range(N_BIG)]

timings, results = {}, {}
for name, fn in [('① sorted   ', lambda: topk_sort(big, K)),
                 ('② heap(k)  ', lambda: topk_heap(big, K)),
                 ('③ quickselect', lambda: topk_quickselect(big, K, rnd_tk))]:
    t0 = time.perf_counter(); out = fn(); dt = time.perf_counter() - t0
    timings[name.strip()], results[name.strip()] = dt, out
    print(f'{name}  {dt*1000:8.2f} ms   top3 = {out[:3]}')
assert len({tuple(v) for v in results.values()}) == 1, '三者结果必须一致'

_QS_VISITS = 0
topk_quickselect(big, K, rnd_tk)
print(f'\nquickselect 的元素访问次数 = {_QS_VISITS:,}  ≈ {_QS_VISITS/N_BIG:.2f}·n  ← 线性')
assert _QS_VISITS < 12 * N_BIG, '随机主元下期望约 2n~4n；12n 是很宽松的上界'

print(f'\n理论比值 log2(n)/log2(k) = {np.log2(N_BIG)/np.log2(K):.2f}x  '
      f'（堆相对全排序的渐进优势）')
print('观察：③ 渐进上是 O(n) 最优，但 ① 走的是 C 层 Timsort，常数小一到两个数量级。')
print('结论：**复杂度是常数未知的上界**。面试里要同时说得出「渐进最优」和「实测更快」，')
print('      并且知道 ② 的真正卖点是 O(k) 空间 + 单遍流式，而不是那个 log 因子。')

## 7 · 经典题单：暴力解 vs 最优解随机对拍

每题标注**频率**与**层级**。所有实现控制在 15–40 行、变量名清晰、注明不变量。

In [ ]:
def duel(name, fast, brute, gen, trials=400):
    for _ in range(trials):
        args = gen()
        f, b = fast(*args), brute(*args)
        assert f == b, (name, args, f, b)
    print(f'  ✅ {name:<34} {trials} 轮随机对拍一致')


# ── 哈希类 ──
def two_sum_hash(a, t):
    """[高频/必会] 边扫边把「值 → 首次出现的下标」存进 dict。O(n)。"""
    seen = {}
    for j, x in enumerate(a):
        if t - x in seen:
            return (seen[t - x], j)
        if x not in seen:               # 只记首次出现，与暴力解的 tie-break 对齐
            seen[x] = j
    return None

def two_sum_brute(a, t):
    for j in range(len(a)):
        for i in range(j):
            if a[i] + a[j] == t:
                return (i, j)
    return None


def longest_consec_hash(a):
    """[中频/必会] 只从「x-1 不在集合里」的序列起点开始向右数 ⇒ 每个元素至多被访问 2 次，总 O(n)。"""
    s, best = set(a), 0
    for x in s:
        if x - 1 in s:
            continue                    # 不是起点，跳过 —— 这一行是 O(n) 的关键
        y = x
        while y + 1 in s:
            y += 1
        best = max(best, y - x + 1)
    return best

def longest_consec_brute(a):
    if not a:
        return 0
    b = sorted(set(a)); best = cur = 1
    for i in range(1, len(b)):
        cur = cur + 1 if b[i] == b[i - 1] + 1 else 1
        best = max(best, cur)
    return best


def topk_freq_bucket(a, k):
    """[高频/必会] 桶排序版：频次上界是 n ⇒ 可以 O(n) 完成，把 log 因子消掉。"""
    c = Counter(a)
    buckets = [[] for _ in range(len(a) + 1)]
    for v, cnt in c.items():
        buckets[cnt].append(v)
    out = []
    for cnt in range(len(a), 0, -1):
        for v in sorted(buckets[cnt]):  # 桶内排序 ⇒ 并列时的顺序也是确定的
            out.append(v)
            if len(out) == k:
                return out
    return out

def topk_freq_brute(a, k):
    c = Counter(a)
    return [v for v, _ in sorted(c.items(), key=lambda kv: (-kv[1], kv[0]))][:k]


def group_anagrams(words):
    """[中频/必会] 排序后的字符元组做 key。O(Σ L log L)。"""
    g = defaultdict(list)
    for w in words:
        g[tuple(sorted(w))].append(w)
    return sorted(sorted(v) for v in g.values())

def group_anagrams_brute(words):
    groups = []
    for w in words:
        for gr in groups:
            if sorted(gr[0]) == sorted(w):
                gr.append(w); break
        else:
            groups.append([w])
    return sorted(sorted(gr) for gr in groups)


rnd_q = random.Random(42)
print('哈希类：')
duel('两数之和 [高频/必会]', two_sum_hash, two_sum_brute,
     lambda: ([rnd_q.randint(-6, 6) for _ in range(rnd_q.randint(0, 9))], rnd_q.randint(-8, 8)))
duel('最长连续序列 [中频/必会]', longest_consec_hash, longest_consec_brute,
     lambda: ([rnd_q.randint(0, 12) for _ in range(rnd_q.randint(0, 12))],))
duel('前 K 高频元素 [高频/必会]', topk_freq_bucket, topk_freq_brute,
     lambda: ([rnd_q.randint(0, 5) for _ in range(rnd_q.randint(1, 12))], rnd_q.randint(1, 4)))
duel('字母异位词分组 [中频/必会]', group_anagrams, group_anagrams_brute,
     lambda: ([''.join(rnd_q.choice('abc') for _ in range(rnd_q.randint(1, 3)))
               for _ in range(rnd_q.randint(0, 8))],))

In [ ]:
# ── 二分 / 排序类 ──
def search_rotated(a, t):
    """[高频/必会] 旋转有序数组（元素互不相同）。
       不变量：任何时刻 [lo, mid] 与 [mid, hi] 中至少有一半是有序的。"""
    lo, hi = 0, len(a) - 1
    while lo <= hi:
        mid = lo + (hi - lo) // 2
        if a[mid] == t:
            return mid
        if a[lo] <= a[mid]:                     # 左半 [lo, mid] 有序
            if a[lo] <= t < a[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:                                   # 右半 [mid, hi] 有序
            if a[mid] < t <= a[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return -1

def rotated_brute(a, t):
    return a.index(t) if t in a else -1

def gen_rotated():
    n = rnd_q.randint(1, 10)
    base = sorted(rnd_q.sample(range(30), n))
    k = rnd_q.randrange(n)
    return (base[k:] + base[:k], rnd_q.randint(0, 29))


def find_range(a, t):
    """[高频/必会] 元素的首末位置：bs_left 与 bs_right - 1。"""
    l = bs_left(a, t)
    if l == len(a) or a[l] != t:
        return (-1, -1)
    return (l, bs_right(a, t) - 1)

def find_range_brute(a, t):
    idx = [i for i, x in enumerate(a) if x == t]
    return (idx[0], idx[-1]) if idx else (-1, -1)


def find_peak(a):
    """[中频/加分] 山脉数组的峰顶。**数组无序，但谓词 a[mid] < a[mid+1] 单调可用** ——
       这道题是「二分的前提是单调谓词而非有序」的最好证据。"""
    lo, hi = 0, len(a) - 1
    while lo < hi:
        mid = lo + (hi - lo) // 2       # lo < hi ⇒ mid < hi ⇒ mid+1 合法
        if a[mid] < a[mid + 1]:
            lo = mid + 1                # 峰一定在右边
        else:
            hi = mid                    # 峰是 mid 或更左
    return lo

def peak_brute(a):
    return max(range(len(a)), key=lambda i: a[i])

def gen_mountain():
    up, down = rnd_q.randint(1, 5), rnd_q.randint(1, 5)
    return (list(range(up)) + [up] + list(range(up - 1, up - 1 - down, -1)),)


def h_index(cit):
    """[中频/加分] 降序排后找最大的 i 使 c[i-1] >= i。"""
    c = sorted(cit, reverse=True); h = 0
    for i, x in enumerate(c, 1):
        if x >= i:
            h = i
        else:
            break                       # c 递减、i 递增 ⇒ 一旦失败就不会再成立
    return h

def h_index_brute(cit):
    return max(h for h in range(len(cit) + 1) if sum(1 for x in cit if x >= h) >= h)


def sort_colors(a):
    """[中频/加分] 荷兰国旗三路划分 —— 和 quickselect 里的划分是同一段代码。
       不变量：[0,i) == 0，[i,t) == 1，(j,n) == 2。"""
    a = list(a); i, j, t = 0, len(a) - 1, 0
    while t <= j:
        if a[t] == 0:
            a[i], a[t] = a[t], a[i]; i += 1; t += 1
        elif a[t] == 2:
            a[t], a[j] = a[j], a[t]; j -= 1     # 换来的元素还没检查，t 不动
        else:
            t += 1
    return a


print('二分 / 排序类：')
duel('搜索旋转排序数组 [高频/必会]', search_rotated, rotated_brute, gen_rotated)
duel('查找元素首末位置 [高频/必会]', find_range, find_range_brute,
     lambda: (sorted(rnd_q.randint(0, 6) for _ in range(rnd_q.randint(0, 10))),
              rnd_q.randint(-1, 7)))
duel('寻找峰值（山脉数组）[中频/加分]', find_peak, peak_brute, gen_mountain)
duel('H 指数 [中频/加分]', h_index, h_index_brute,
     lambda: ([rnd_q.randint(0, 8) for _ in range(rnd_q.randint(1, 8))],))
duel('颜色分类（荷兰国旗）[中频/加分]', sort_colors, sorted,
     lambda: ([rnd_q.randint(0, 2) for _ in range(rnd_q.randint(0, 12))],))
print('\n✅ 10 道题全部通过暴力解对拍（含第 5、6 节的 LC410 与 Top-K）。')

## ✏️ 练习 1：只背一套模板，其余全部现推

实现四个函数（**不许调用 `bisect`**）：
- `my_bs_left(a, t)` → 第一个 `>= t` 的下标（等价 `bisect_left`）。半开区间 `[lo, hi)`，
  不变量：`a[:lo]` 全 `< t` 且 `a[hi:]` 全 `>= t`
- `my_bs_right(a, t)` → 第一个 `> t` 的下标（等价 `bisect_right`）。与上面**只差一个 `=`**
- `count_equal(a, t)` → `t` 的出现次数。**只能用上面两个函数**（不许 `count` / 循环）
- `last_le(a, t)` → 最后一个 `<= t` 的下标，不存在返回 `-1`。**只能用上面两个函数**

In [ ]:
def my_bs_left(a, t):
    # TODO: 半开区间 [lo, hi)，lo=0, hi=len(a)
    #       a[mid] < t  -> lo = mid + 1  （必须 +1，否则死循环）
    #       否则         -> hi = mid
    raise NotImplementedError

def my_bs_right(a, t):
    # TODO: 与 my_bs_left 只差把 `a[mid] < t` 换成 `a[mid] <= t`
    raise NotImplementedError

def count_equal(a, t):
    # TODO: 用换算表
    raise NotImplementedError

def last_le(a, t):
    # TODO: 用换算表
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
_r1 = random.Random(3)
for _ in range(3000):
    n = _r1.randint(0, 10)
    a = sorted(_r1.randint(0, 5) for _ in range(n))
    t = _r1.randint(-1, 6)
    assert my_bs_left(a, t)  == bisect.bisect_left(a, t),  ('left', a, t)
    assert my_bs_right(a, t) == bisect.bisect_right(a, t), ('right', a, t)
    assert count_equal(a, t) == a.count(t), ('count', a, t)
    assert last_le(a, t) == max([i for i, x in enumerate(a) if x <= t], default=-1), ('le', a, t)
# 空数组与全相同数组这两个边界必须单独点名
assert my_bs_left([], 5) == 0 and my_bs_right([], 5) == 0 and last_le([], 5) == -1
assert my_bs_left([2, 2, 2], 2) == 0 and my_bs_right([2, 2, 2], 2) == 3
assert count_equal([2, 2, 2], 2) == 3 and last_le([2, 2, 2], 2) == 2
print('✅ 练习 1 通过：3000 组随机对拍 + 空/全相同边界，四个函数与标准库完全一致。')
print('   记住这条：面试里只背 bs_left，右边界 / 计数 / 精确查找全部由它现推。')

## ✏️ 练习 2：在答案上二分

实现 `min_ship_capacity(weights, days)`（LeetCode 1011）：
包裹必须**按给定顺序**在 `days` 天内运完，每天装的包裹重量和不超过船的运载能力，求**最小运载能力**。

步骤：
1. 写 `need(cap)`：贪心地按顺序装，装不下就开新的一天，返回需要几天
2. `lo = max(weights)`（一天至少要装得下最重的那个包裹），`hi = sum(weights)`（一天全装完一定可行）
3. 套模板 B：`need(mid) <= days` → `hi = mid`，否则 `lo = mid + 1`

**先在心里说清楚：为什么 `need(cap) <= days` 是关于 `cap` 单调的？**

In [ ]:
def min_ship_capacity(weights, days):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert min_ship_capacity([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5) == 15
assert min_ship_capacity([3, 2, 2, 4, 1, 4], 3) == 6
assert min_ship_capacity([1, 2, 3, 1, 1], 4) == 3
assert min_ship_capacity([7], 1) == 7

def _brute_capacity(w, d):
    """暴力对照：从 max(w) 开始线性往上试第一个可行的 cap。"""
    def need(cap):
        cnt, cur = 1, 0
        for x in w:
            if cur + x <= cap:
                cur += x
            else:
                cnt += 1; cur = x
        return cnt
    cap = max(w)
    while need(cap) > d:
        cap += 1
    return cap

_r2 = random.Random(11)
for _ in range(300):
    n = _r2.randint(1, 9)
    w = [_r2.randint(1, 9) for _ in range(n)]
    d = _r2.randint(1, n)
    assert min_ship_capacity(w, d) == _brute_capacity(w, d), (w, d)
# 单调性自检：可行性一旦为真，就不会再变回假
_w, _d = [3, 2, 2, 4, 1, 4], 3
_ans = min_ship_capacity(_w, _d)
_feas = [_brute_capacity(_w, _d) <= c for c in range(max(_w), sum(_w) + 1)]
assert _feas == sorted(_feas), 'check 必须单调，否则不能二分'
print(f'✅ 练习 2 通过：300 组随机数据与线性扫描一致；最小运载能力 = {_ans}')
print('   同一套模板还能解：爱吃香蕉的珂珂、制作 m 束花、分割数组的最大值、')
print('   以及工程里的「给定延迟预算求最大分辨率」「给定召回下限求最高阈值」。')

## ✏️ 练习 3：quickselect

实现 `my_quickselect(a, k, rnd)` 返回列表 `a` 中**第 k 小**的值（`k` 从 0 开始）。要求：
- **随机主元**（用 `rnd.randint(lo, hi)`）—— 否则有序输入会退化成 O(n²)
- **三路划分**（荷兰国旗）—— 否则大量重复元素时也会退化
- **不修改传入的 `a`**（内部先 `list(a)` 复制）
- 不许调用 `sorted` / `heapq` / `min` / `max`

In [ ]:
def my_quickselect(a, k, rnd):
    # TODO：
    #   a = list(a); lo, hi = 0, len(a)-1
    #   循环：p = a[rnd.randint(lo, hi)]
    #        三路划分成 [lo,i) < p，[i,j] == p，(j,hi] > p
    #        k < i  -> hi = i-1 ；k > j -> lo = j+1 ；否则 return p
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
_r3 = random.Random(5)
assert my_quickselect([3, 1, 2], 0, _r3) == 1
assert my_quickselect([3, 1, 2], 2, _r3) == 3
assert my_quickselect([5, 5, 5, 5], 2, _r3) == 5       # 全相同：三路划分的关键用例
assert my_quickselect([9], 0, _r3) == 9                 # 单元素

_src = [7, 7, 1, 9, 3, 3, 3, 8]
for _k in range(len(_src)):
    assert my_quickselect(_src, _k, _r3) == sorted(_src)[_k], _k
assert _src == [7, 7, 1, 9, 3, 3, 3, 8], '不许修改调用方传入的列表'

for _ in range(300):
    n = _r3.randint(1, 25)
    a = [_r3.randint(0, 8) for _ in range(n)]           # 大量重复
    k = _r3.randrange(n)
    assert my_quickselect(a, k, _r3) == sorted(a)[k], (a, k)

# 有序输入也必须是线性的（随机主元的意义）
_sorted_in = list(range(4000))
_t0 = time.perf_counter()
assert my_quickselect(_sorted_in, 2000, _r3) == 2000
_dt = time.perf_counter() - _t0
print(f'✅ 练习 3 通过：300 组随机数据 + 全相同 + 单元素；'
      f'4000 个已排序元素上取中位数只用了 {_dt*1000:.1f} ms（未退化）')

## ✏️ 练习 4：让 NMS 的输入成为确定性的

实现 `nms_order(scores, boxes)`：返回下标的排序结果，满足
- 主序：**分数降序**
- 并列时用 `(x1, y1, 原始下标)` 依次 tie-break —— 保证键是**全序**（任意两个不同元素的键都不相等）

这样一来，「排序是否稳定」就不再重要，因为根本不会出现并列。

In [ ]:
def nms_order(scores, boxes):
    # TODO: return sorted(range(n), key=lambda i: (...))
    #       注意把 numpy 标量转成 float / int，避免比较时的类型意外
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测（手算）——
_sc = np.array([0.9, 0.9, 0.7, 0.9])
_bx = np.array([[10., 10., 20., 20.],     # 0
                [ 5., 30., 15., 40.],     # 1
                [50., 50., 60., 60.],     # 2
                [10.,  5., 20., 15.]])    # 3
_o = nms_order(_sc, _bx)
assert [round(float(_sc[i]), 3) for i in _o] == [0.9, 0.9, 0.9, 0.7], _o
# 并列的 0/1/3 按 (x1, y1) 升序：(5,30) < (10,5) < (10,10)
assert _o == [1, 3, 0, 2], _o

# 换个输入顺序，排序后的「框序列」必须逐字节一致
_perm = [2, 0, 3, 1]
_o2 = nms_order(_sc[_perm], _bx[_perm])
assert np.array_equal(_bx[_perm][_o2], _bx[_o]), '全序键 ⇒ 输出与输入顺序无关'

# 接上第 2 节的 NMS：全序键下，打乱输入不改变保留的框
_k1 = sorted(nms(BX, nms_order(SC, BX)))
_k2 = sorted([2, 1, 0][j] for j in nms(BX[[2, 1, 0]], nms_order(SC[[2, 1, 0]], BX[[2, 1, 0]])))
assert _k1 == _k2 == [0, 2], (_k1, _k2)
print('✅ 练习 4 通过：全序排序键让整条 NMS 流水线成为确定性函数。')
print('   面试里被问「你的代码是确定性的吗」，这就是标准答案。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def my_bs_left(a, t):
    lo, hi = 0, len(a)                    # 不变量：a[:lo] < t <= a[hi:]
    while lo < hi:
        mid = lo + (hi - lo) // 2
        if a[mid] < t:
            lo = mid + 1
        else:
            hi = mid
    return lo

def my_bs_right(a, t):
    lo, hi = 0, len(a)                    # 不变量：a[:lo] <= t < a[hi:]
    while lo < hi:
        mid = lo + (hi - lo) // 2
        if a[mid] <= t:                   # ← 唯一的差别
            lo = mid + 1
        else:
            hi = mid
    return lo

def count_equal(a, t):
    return my_bs_right(a, t) - my_bs_left(a, t)

def last_le(a, t):
    return my_bs_right(a, t) - 1

In [ ]:
# 练习 2 参考答案
def min_ship_capacity(weights, days):
    def need(cap):
        # 贪心：能装就装。可以证明这是最优的（提前开新的一天不会更好）
        cnt, cur = 1, 0
        for x in weights:
            if cur + x <= cap:
                cur += x
            else:
                cnt += 1; cur = x
        return cnt

    # 单调性：cap 增大 ⇒ 每天能装的不减 ⇒ need(cap) 单调不增 ⇒ need(cap) <= days 单调
    lo, hi = max(weights), sum(weights)
    while lo < hi:
        mid = lo + (hi - lo) // 2
        if need(mid) <= days:
            hi = mid
        else:
            lo = mid + 1
    return lo

In [ ]:
# 练习 3 参考答案
def my_quickselect(a, k, rnd):
    a = list(a)                                  # 不动调用方的数据
    lo, hi = 0, len(a) - 1
    while True:
        p = a[rnd.randint(lo, hi)]               # 随机主元
        i, j, t = lo, hi, lo
        # 划分不变量：[lo,i) < p，[i,t) == p，(j,hi] > p，[t,j] 未检查
        while t <= j:
            if a[t] < p:
                a[i], a[t] = a[t], a[i]; i += 1; t += 1
            elif a[t] > p:
                a[t], a[j] = a[j], a[t]; j -= 1  # 换来的元素还没检查，t 不动
            else:
                t += 1
        if k < i:
            hi = i - 1
        elif k > j:
            lo = j + 1
        else:
            return p                             # 落在等于主元的段内，直接出答案

In [ ]:
# 练习 4 参考答案
def nms_order(scores, boxes):
    n = len(scores)
    return sorted(range(n), key=lambda i: (-float(scores[i]),
                                           float(boxes[i, 0]),
                                           float(boxes[i, 1]),
                                           i))

---
## 🧪 真实工程胶囊：哈希 / 排序 / 二分的速查卡

In [ ]:
RECIPE = r'''
# ======================================================================
# 面试与生产两用速查卡 · 哈希 / 排序 / 二分            （C62 模块 02）
# ======================================================================

# ---------- 1. 只背这一套二分，其余全部现推 ----------
def bisect_left_(a, t):
    lo, hi = 0, len(a)              # 半开区间 [lo, hi)，hi = 第一个被排除的位置
    while lo < hi:                  # 不变量：a[:lo] < t <= a[hi:]
        mid = lo + (hi - lo) // 2   # 下取整 => lo <= mid < hi => 两支都严格缩区间
        if a[mid] < t: lo = mid + 1 # 必须 +1，否则度量函数不递减 -> 死循环
        else:          hi = mid
    return lo                       # 终止：至多 ceil(log2 n) 轮

# 换算表（不要再背第二套模板）
#   bisect_right = 把上面的 `<` 换成 `<=`
#   出现次数      = bisect_right - bisect_left
#   最后一个 <= t = bisect_right - 1
#   最后一个 <  t = bisect_left  - 1
#   精确查找      = i = bisect_left(a,t);  a[i] == t ? i : -1
# 求右边界若非要写成 lo = mid，则 mid 必须上取整：mid = lo + (hi-lo+1)//2

# ---------- 2. 在答案上二分（最小化最大值 / 最大化最小值） ----------
# 三个识别信号：(1) 题面是"最小化最大值 / 至少需要多少"
#               (2) check(x) 好写（通常一遍 O(n) 贪心）
#               (3) check 单调（x 可行 => x+1 可行）   <-- 必须先说出这一句
def min_feasible(lo, hi, check):                     # 整数答案空间
    while lo < hi:
        mid = lo + (hi - lo) // 2
        if check(mid): hi = mid
        else:          lo = mid + 1
    return lo

def min_feasible_float(lo, hi, check, iters=100):    # 浮点：固定次数，绝不用 while
    for _ in range(iters):
        mid = (lo + hi) / 2
        if check(mid): hi = mid
        else:          lo = mid
    return hi

# 工程同构（同一个算法，换个名字）：
#   给定延迟预算 -> 最大输入分辨率      check(r) = latency(r) <= budget
#   给定召回下限 -> 最高 score 阈值      check(th) = recall(th) >= 0.95
#   切片推理     -> 最小安全重叠率       check(ov) = 任意目标完整落入某片
#   带宽预算     -> 难例回传触发阈值     check(th) = 触发量 <= 预算
# 陷阱：单调性没验证就二分。延迟对分辨率并非严格单调（kernel 选择会有反常点），
#      先扫粗网格确认单调，再在单调段内二分。

# ---------- 3. 确定性 NMS：把排序键做成全序 ----------
# 症状：一致性对拍时 1000 帧里有几帧"差一个框"，位置差几像素，分数完全相同。
# 根因：分数并列（INT8 量化后分类头只有 256 档）+ tie-break 在各实现间不一致：
#       np.argsort 默认 quicksort 不稳定 / torch.sort 默认不保证 /
#       TensorRT TopK 无文档化保证（GPU 归约顺序决定）。
def stable_order(scores, boxes, class_ids):
    n = len(scores)
    return sorted(range(n), key=lambda i: (-float(scores[i]),
                                           int(class_ids[i]),
                                           float(boxes[i][0]), float(boxes[i][1]), i))
# 有了全序键，"排序是否稳定"就不再重要 —— 因为根本不会出现并列。
# 同样的道理适用于任何贪心：区间调度、并查集聚类、匈牙利匹配的候选枚举。

# ---------- 4. 分桶评测：绝不用浮点当 dict 的 key ----------
import bisect as _bs
EDGES  = [0, 16, 32, 64, 128, 256]           # sqrt(面积) 的桶边界，左闭右开
LABELS = ['<16', '16-32', '32-64', '64-128', '128-256', '>=256']
def bucket_of(v):
    return min(max(_bs.bisect_right(EDGES, v) - 1, 0), len(LABELS) - 1)
# 反例：buckets[round(iou, 2)] += 1
#   round(0.035,2)=0.04 向上，round(0.045,2)=0.04 向下 —— 方向由二进制表示决定，
#   两台机器上差 1e-16 的 IoU 就可能落进不同的桶，评测报告对不上。

# ---------- 5. Top-K 选型（先问四个问题） ----------
#   k 与 n 的量级？ 数据能一次进内存？ 要有序输出？ 能改原数组？
#   k << n 且流式     -> heapq.nlargest / 大小 k 的最小堆   O(n log k)  空间 O(k)
#   n 极大、不要顺序  -> quickselect（**随机主元 + 三路划分**）O(n) 期望
#   k 接近 n 或要有序 -> sorted                              O(n log n)
#   实测提醒：CPython 的 sorted 是 C 层 Timsort，n ~ 1e5 时常比手写 quickselect 更快。
#   检测后处理的 nms_pre = top-1000：不是为精度，是把 NMS 的最坏耗时钉死（见 C53-04）。

# ---------- 6. 交卷前 30 秒自检 ----------
#   [ ] 二分：说出用的是闭区间还是半开区间；用"区间只剩 1-2 个元素"手推一遍
#   [ ] 二分：mid 写成 lo + (hi-lo)//2（C++/Java 防溢出，JDK 曾为此挂了九年）
#   [ ] 答案二分：显式说明 check 为什么单调、lo/hi 各自的物理含义
#   [ ] 哈希：报复杂度时说清"查找期望 O(1) / 最坏 O(n) / 插入均摊 O(1)"
#   [ ] 排序：下游若是贪心（NMS / 区间调度），排序键必须是全序
#   [ ] 边界五连：空输入、单元素、全相同、目标不存在、目标落在两端
'''
print(RECIPE)
for _tok in ['bisect_left_', 'min_feasible_float', 'stable_order', 'bucket_of',
             'quickselect', '边界五连']:
    assert _tok in RECIPE, _tok
print('\n（速查卡建议面试前一天过一遍；前 3 节是可直接复制进项目的代码）')

### 小结

1. **二分的前提不是「数组有序」，是「存在一个可切分的单调谓词」。**
   写代码前先把 `check` 写出来并确认单调 —— 做到这一步，「在答案上二分」就是同一套模板的直接应用，
   而「寻找峰值」这种无序数组也能二分就不再神秘。

2. **只背 `bisect_left` 一套模板，其余用换算表现推。** 少一套模板就少一类 bug。
   死循环的通用判据只有一条：**mid 落在哪一半，那一半的更新就必须跨过 mid**；
   正确性靠「不变量三段式 + 严格递减的整数度量」来证明，而不是靠多做题找感觉。

3. **报复杂度要说清口径。** dict 查找是「期望 O(1)、最坏 O(n)」，插入是「均摊 O(1)」；
   而均摊 O(1) 意味着某一次会突然很慢 —— 在每帧都要跑的关键路径上，这是要预分配来规避的。

4. **渐进复杂度是常数未知的上界。** 本 notebook 实测：$n=10^5$ 时 C 层 Timsort 的全排序
   比纯 Python 的 O(n) quickselect 还快。面试里同时说得出「渐进最优」和「实测更快」，
   比背一个复杂度符号有价值得多。

5. **凡是贪心，输出就依赖排序顺序；凡是依赖排序顺序，排序键就必须是全序。**
   NMS 是最典型的例子：分数并列（INT8 量化后只有 256 档）+ 各实现 tie-break 不一致
   = 训练端与车端「差一个框」的疑难杂症。修法只有一行：把键扩成 `(-score, cls, x1, y1, idx)`。